# Gordion planetary dark matter phase space

This notebook samples dark-matter positions and velocities in celestial-body models, follows velocity-changing nuclear scatters, and records first-capture phase points. Gordion's self-contained capture model implements the multi-regime treatment of Leane & Smirnov (2023), also released as Asteria v1.0. Neither Gordion nor this notebook requires Asteria.

The built-in Earth, Moon, and Mars interiors are explicit piecewise-uniform shells. Catalog exoplanets use a two-layer iron-core/silicate-mantle model with an assumed CMF of 0.33. The archive's mass provenance is retained because some values are minimum masses or mass-radius estimates. Lagrange-point samples use a local halo distribution in the binary's co-rotating frame. They do not imply stable trapping.

The trajectory model treats target nuclei as cold and stationary and scattering as isotropic and elastic. It omits thermal target motion, inelastic channels, annihilation, and post-capture thermalization. The unbound interior distribution uses energy conservation with a straight-ray direction mapping. The capture-rate module separately follows the published homogeneous-composition formalism.

## 1. Environment Setup and Reproducible Configuration

Set a deterministic seed, discover the repository root, configure Gordion's data paths, and select the plotting and transport settings.

## 2. Gordion Data Model for 6D Phase Space

Represent each state as $(x,y,z,v_x,v_y,v_z)$, with weights and particle identifiers retained alongside trajectory and capture metadata.

## 3. Gravitational Potential and Interior Structure Modules

Construct Earth, Moon, Mars, and generic exoplanets from spherical density shells. Vary the core mass fraction and preserve each model's total mass and radius.

## 4. Dark Matter Orbit Integration in Full 6D

Sample a shifted Maxwellian halo, gravitationally focus incoming particles, and propagate their six-dimensional states through each spherical potential.

## 5. Scattering Physics and Velocity-Kick Updates

At each integration step, draw nuclear scatters from the local optical depth and apply isotropic elastic two-body velocity changes. Optional sanity checks verify momentum and energy conservation at collisions.

## 6. Capture Criteria and Bound-Population Accounting

Classify a scattered particle as captured when its specific orbital energy becomes negative inside the body. Retain the capture position, velocity, scatter count, and flux-weighted capture rate.

## 7. Gordion Multi-Regime Capture Model

Calculate geometric focusing, nuclear cross-section scaling, optical depth, single and multiple scatters, weak/intermediate/strong regimes, finite-target stopping, and the light-DM reflection fit with `CaptureModel`. Composition is instance-scoped.

## 8. Reference Tests and Optional Asteria Comparison

The unit tests compare Gordion against stored Jupiter reference rates without importing Asteria. Only `examples/asteria_benchmark/run.py` imports the released Asteria Python v1.0 implementation for a live comparison. It covers weak, multiscatter, and high-optical-depth cases with both cross-section conventions.

## 9. Example: Velocity Distribution Inside Earth

Sample the Earth interior and scatter trajectories. Compare unbound and first-capture velocity distributions at selected depths.

## 10. Example: Velocity Distribution Inside Moon

Repeat the same calculation with the lunar layered profile and compare the resulting velocity and capture distributions.

## 11. Example: Velocity Distribution Inside Mars

Run Mars with scattering enabled and inspect its spatial and velocity distributions alongside the scattering-free samples.

## 12. Example: All Catalog Exoplanets

Download or reuse the NASA Exoplanet Archive snapshot. Estimate interior phase-space samples for every catalog row with positive mass and radius, keeping its mass provenance.

## 13. Example: Variable CMFs and Mass-Density Profiles

Compare Earth-size rocky planets, super-Earths, and sub-Neptunes across CMFs of 0.0–0.7. Plot density and escape-speed profiles and calculate capture rates under each composition.

## 14. Example: Co-rotating Lagrange Points

Solve L1–L5 for Sun–Earth, Sun–Jupiter, HD 108236, TRAPPIST-1, and WASP-121. Sample local six-dimensional halo states and transform velocities to each rotating frame.

## 15. Batch Execution, Result Storage, and Visualization

The same analyses are available through `examples/planetary_phase_space/run.py`. Timestamped figures are written directly to the example's `visuals/` directory. Capture tables, the exoplanet archive snapshot, and full six-dimensional catalog samples are stored under Gordion's ignored `data/` directory.

In [ ]:
import os
from datetime import datetime, timezone
from pathlib import Path

repo_root = Path.cwd().resolve()
while not (repo_root / "pyproject.toml").is_file() and repo_root.parent != repo_root:
    repo_root = repo_root.parent
if not (repo_root / "gordion" / "__init__.py").is_file():
    raise RuntimeError("Run this notebook from the Gordion repository or one of its subdirectories.")
os.environ.setdefault("GORDION_PATH", str(repo_root))
os.environ["TDPY_VERBOSITY"] = "0"

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display

from gordion.bodies import solar_system_bodies, two_layer_exoplanet
from gordion.capture import CaptureModel
from gordion.catalog import fetch_confirmed_exoplanets
from gordion.lagrange import BinarySystem, lagrange_points, sample_lagrange_phase_space
from gordion.phase_space import (
    HaloModel,
    sample_unbound_phase_space,
    simulate_capture_trajectories,
)

example_dir = repo_root / "examples" / "planetary_phase_space"
visuals_dir = example_dir / "visuals"
visuals_dir.mkdir(parents=True, exist_ok=True)
run_tag = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")
seed = 17
rng = np.random.default_rng(seed)
halo = HaloModel(density_gev_cm3=0.4, speed_scale_m_s=220000.0,
                 galactic_escape_speed_m_s=544000.0,
                 body_velocity_m_s=(232000.0, 0.0, 0.0))

def save_figure(figure, stem):
    path = visuals_dir / f"{stem}_{run_tag}.png"
    relative = path.relative_to(repo_root)
    print(f"Writing to {relative}...")
    figure.savefig(path, dpi=240, bbox_inches="tight")
    display(figure)
    plt.close(figure)
    return relative

print("Random seed:", seed)
print("Gordion runtime path configured; notebook outputs use relative paths.")

In [ ]:
# Earth, Moon, and Mars use explicit spherical shells.
solar_bodies = solar_system_bodies()
profile_rows = []
radius_fraction = np.linspace(0.0, 1.0, 240)
figure, axes = plt.subplots(1, 2, figsize=(11, 4.3), constrained_layout=True)
for name, body in solar_bodies.items():
    radius_m = radius_fraction * body.radius_m
    density = body.density_kg_m3(radius_m)
    escape_km_s = body.escape_speed_m_s(radius_m) / 1000.0
    axes[0].plot(radius_fraction, density, label=name)
    axes[1].plot(radius_fraction, escape_km_s, label=name)
    profile_rows.append({
        "body": name,
        "mass_kg": body.mass_kg,
        "radius_m": body.radius_m,
        "surface_density_kg_m3": body.density_kg_m3(body.radius_m * 0.999),
        "center_escape_speed_km_s": body.escape_speed_m_s(0.0) / 1000.0,
        "surface_escape_speed_km_s": body.escape_speed_m_s(body.radius_m) / 1000.0,
        "element_count": len(body.composition),
    })
axes[0].set(xlabel="Radius / body radius", ylabel="Density [kg m$^{-3}$]",
            title="Adopted shell density")
axes[1].set(xlabel="Radius / body radius", ylabel="Escape speed [km s$^{-1}$]",
            title="Escape speed from the radial potential")
axes[0].legend(frameon=False)
display(profile_rows)
save_figure(figure, "solar_body_profiles")

In [ ]:
from gordion.bodies import JUPITER_MASS_KG, JUPITER_RADIUS_M

jupiter_capture = CaptureModel(((0.75, 1.0), (0.25, 4.0)))
reference_rate = jupiter_capture.multicapture(
    JUPITER_RADIUS_M, JUPITER_MASS_KG, 0.4, 270000.0, 0.94, 1e-38, 10.0
)
print(f"Gordion Jupiter capture rate: {reference_rate:.4e} captures/s")

mass_grid_gev = np.logspace(-2, 4, 24)
sigma_reference_cm2 = 1e-38
capture_rows = []
figure, axis = plt.subplots(figsize=(7.2, 4.8), constrained_layout=True)
for name, body in solar_bodies.items():
    model = CaptureModel(body.composition)
    rates = np.array([
        model.calculate(body.radius_m, body.mass_kg, 0.4, 270000.0,
                        mass_gev, sigma_reference_cm2, 0.94).rate_per_second
        for mass_gev in mass_grid_gev
    ])
    positive = rates > 0.0
    axis.loglog(mass_grid_gev[positive], rates[positive], label=name)
    capture_rows.extend({"body": name, "dm_mass_gev": mass_gev,
                         "nucleon_cross_section_cm2": sigma_reference_cm2,
                         "capture_rate_s-1": rate}
                        for mass_gev, rate in zip(mass_grid_gev, rates))
axis.set(xlabel="Dark-matter mass [GeV]", ylabel="Capture rate [s$^{-1}$]",
         title="Gordion multi-regime capture rate")
axis.legend(frameon=False)
import csv
from gordion.paths import get_data_path
capture_dir = get_data_path() / "planetary_phase_space"
capture_dir.mkdir(parents=True, exist_ok=True)
capture_path = capture_dir / f"solar_system_capture_{run_tag}.csv"
print(f"Writing to {capture_path.relative_to(repo_root)}...")
with capture_path.open("w", newline="", encoding="utf-8") as stream:
    writer = csv.DictWriter(stream, fieldnames=list(capture_rows[0]))
    writer.writeheader()
    writer.writerows(capture_rows)
save_figure(figure, "solar_system_capture")

In [ ]:
# Sample unbound 6D states and follow scatter-modified trajectories in each body.
from gordion.phase_space import sample_unbound_phase_space, simulate_capture_trajectories

transport_rows = []
figure, axes = plt.subplots(3, 2, figsize=(11, 10), constrained_layout=True)
body_speeds_m_s = {"Earth": 29780.0, "Moon": 30800.0, "Mars": 24100.0}
for row, (name, body) in enumerate(solar_bodies.items()):
    body_halo = HaloModel(body_velocity_m_s=(232000.0, body_speeds_m_s[name], 0.0))
    free = sample_unbound_phase_space(body, 768, body_halo, seed=seed + row)
    transported = simulate_capture_trajectories(
        body, 96, 10.0, 1e-34, body_halo, step_fraction=0.04,
        max_steps=2500, sample_stride=5, seed=seed + row, sanity_checks=True,
    )
    free_speed = np.linalg.norm(free.velocity_m_s, axis=1) / 1000.0
    axes[row, 0].hist(free_speed, bins=28, weights=free.weight, density=True,
                      histtype="step", color="#007360", label="Unbound")
    path = transported.trajectory_samples
    if len(path.weight):
        axes[row, 0].hist(np.linalg.norm(path.velocity_m_s, axis=1) / 1000.0,
                          bins=28, weights=path.weight, density=True,
                          histtype="step", color="#cc6677", label="Scattering paths")
        axes[row, 1].scatter(path.position_m[:, 0] / body.radius_m,
                             path.velocity_m_s[:, 0] / 1000.0,
                             s=5, alpha=0.35, color="#176B87", rasterized=True)
    if len(transported.capture_phase_space.weight):
        axes[row, 0].hist(np.linalg.norm(transported.capture_phase_space.velocity_m_s, axis=1) / 1000.0,
                          bins=14, weights=transported.capture_phase_space.weight,
                          density=True, histtype="step", color="#332288",
                          label="First-capture states")
    axes[row, 0].set(xlabel="Speed [km s$^{-1}$]", ylabel="Weighted density", title=name)
    axes[row, 0].legend(frameon=False, fontsize=8)
    axes[row, 1].set(xlabel="$x/R$", ylabel="$v_x$ [km s$^{-1}$]",
                     title="Position-velocity projection")
    transport_rows.append({"body": name,
                           "particles": len(transported.particle_status),
                           "captured": int(np.sum(transported.particle_status == "captured")),
                           "escaped": int(np.sum(transported.particle_status == "escaped")),
                           "capture_rate_estimate_s-1": transported.capture_rate_per_second})
display(transport_rows)
save_figure(figure, "earth_moon_mars_scattering_phase_space")

In [ ]:
# Sweep the complete retained NASA Exoplanet Archive snapshot.
import runpy

catalog_path = Path("data/exoplanet_archive/planetary_systems.csv")
print(f"Reading from {catalog_path}..." if catalog_path.is_file()
      else "Reading from NASA Exoplanet Archive TAP service...")
exoplanets = fetch_confirmed_exoplanets()
print("Planets with positive catalog mass and radius:", len(exoplanets))
example_tools = runpy.run_path(str(example_dir / "run.py"))
processed_planets = example_tools["_plot_exoplanets"](
    exoplanets, samples_per_planet=24, seed=seed, tag=run_tag
)
print("Six-dimensional planet samples generated:", processed_planets * 24)
display(exoplanets[:5])

In [ ]:
# Vary CMF and density profiles at fixed mass and radius.
cmf_models = []
for mass_earth, radius_earth, label in ((1.0, 1.0, "Earth-size rocky"),
                                        (5.0, 1.5, "Super-Earth"),
                                        (8.0, 2.5, "Sub-Neptune")):
    for cmf in (0.0, 0.1, 0.3, 0.5, 0.7):
        model = two_layer_exoplanet(label, mass_earth, radius_earth, cmf)
        cmf_models.append({"type": label, "mass_earth": mass_earth,
                           "radius_earth": radius_earth, "CMF": cmf,
                           "mean_density_kg_m3": model.mass_kg / (4.0 * np.pi * model.radius_m**3 / 3.0),
                           "central_escape_speed_km_s": model.escape_speed_m_s(0.0) / 1000.0})
display(cmf_models)
example_tools["_plot_cmf_grid"](seed=seed, tag=run_tag)

In [ ]:
# Co-rotating L1-L5 distributions for Solar System and catalog binaries.
from gordion.bodies import JUPITER_MASS_KG, SUN_MASS_KG

solar_a_m = 1.495978707e11
jupiter_a_m = 5.2044 * solar_a_m
lagrange_systems = [
    BinarySystem(SUN_MASS_KG, solar_bodies["Earth"].mass_kg, solar_a_m, "Sun", "Earth"),
    BinarySystem(SUN_MASS_KG, JUPITER_MASS_KG, jupiter_a_m, "Sun", "Jupiter"),
]
print("Circular restricted-three-body samples include Earth-Sun and Jupiter-Sun.")
print("The batch runner also includes HD 108236, TRAPPIST-1, and WASP-121 when catalog orbit metadata are available.")
lagrange_system_count = example_tools["_plot_lagrange_points"](
    exoplanets, samples=256, seed=seed, tag=run_tag
)
print("Planetary systems sampled:", lagrange_system_count)

## Batch execution and stored products

The equivalent command-line run is:

```console
python examples/planetary_phase_space/run.py --mode all
```

Use ``--offline`` after the archive snapshot is cached, ``--refresh`` to download an updated NASA table, ``--limit-planets`` for a shorter development run, and ``--sanity-checks`` to test each elastic-scatter update. Every run uses a UTC timestamp in its output filenames, so earlier figures and tables remain available. Data inputs and derived CSV tables stay under Gordion's ignored ``data/`` tree. Figures are written directly to this example's ``visuals/`` directory.

In [ ]:
# Review generated products without exposing machine-specific paths.
relative_products = sorted(
    path.relative_to(repo_root).as_posix()
    for path in visuals_dir.iterdir()
    if path.is_file()
)
print("Example outputs:")
for product in relative_products[-24:]:
    print(product)
print("Full catalog rows retained:", len(exoplanets))
print("Each saved catalog sample contains x, y, z, vx, vy, vz, and a Liouville weight.")